### ANN & Naive Bayes Classification — Car Price Range

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, ConfusionMatrixDisplay)

### 1. Load Data

In [0]:
df = pd.read_csv("car_prices.csv")


### 2. Drop rows with key missing values


In [0]:
df.dropna(subset=['sellingprice', 'condition', 'odometer', 'mmr'], inplace=True)


### 3. Engineer Target Variable

In [0]:
def assign_price_range(price):
    if price < 10000:
        return "Budget"
    elif price < 30000:
        return "Mid"
    else:
        return "Luxury"

df['price_range'] = df['sellingprice'].apply(assign_price_range)

### 4. Encode Categorical Features

In [0]:
le = LabelEncoder()
for col in ['make', 'body', 'transmission', 'color']:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

### 5. Define Features and Target

In [0]:
features = ['year', 'condition', 'odometer', 'mmr',
            'make_enc', 'body_enc', 'transmission_enc', 'color_enc']
X = df[features]
y = df['price_range']

### 6. Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

###  7. Scale Features for ANN

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### 8. Naive Bayes

In [0]:
gnb = GaussianNB(var_smoothing=1e-9)
gnb.fit(X_train, y_train)
y_pred_nb = gnb.predict(X_test)

print("=== Naive Bayes Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(classification_report(y_test, y_pred_nb))

### 9. ANN (MLP Classifier)

In [0]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

print("\n=== ANN (MLP) Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(classification_report(y_test, y_pred_mlp))

Runs about 9 minutes total

### 10. Confusion Matrices

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_nb),
    display_labels=gnb.classes_).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Naive Bayes — Confusion Matrix")

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_mlp),
    display_labels=mlp.classes_).plot(ax=axes[1], colorbar=False)
axes[1].set_title("ANN (MLP) — Confusion Matrix")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

### 11. Accuracy Comparison Bar Chart

In [0]:
models = ['Naive Bayes', 'ANN (MLP)']
accuracies = [accuracy_score(y_test, y_pred_nb),
              accuracy_score(y_test, y_pred_mlp)]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, accuracies, color=['#4C72B0', '#DD8452'], width=0.4)
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{acc:.4f}", ha='center', fontsize=11)
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

### 12. Class Distribution

In [0]:
plt.figure(figsize=(6, 4))
df['price_range'].value_counts().plot(kind='bar', color=['#4C72B0','#DD8452','#55A868'])
plt.title("Price Range Class Distribution")
plt.xlabel("Price Range")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()